# Assignment 9
Develop a system for identifying the key gene for cancer disease. For this purpose follow the
following steps: 
 - a. Download the gene expression cancer RNA-Seq Data Set. It comprises the gene expression
spanning over different types of cancer patients. 
 - b. Apply the CLIQUE clustering techniques for finding clusters prevailing in different dimensions
starting with 2 dimensions at a time.
 - c. Key genes are those gene that are contributing to clusters across different dimensions. So the
set of key gene can be obtained by taking the intersection over the set of genes representing clusters
across different dimensions.

## Load Data

In [22]:
import pandas as pd
data_df = pd.read_csv('/kaggle/input/gene-expression-cancer-rna-seq/TCGA-PANCAN-HiSeq-801x20531/TCGA-PANCAN-HiSeq-801x20531/data.csv')
data_df.head()

,Unnamed: 0,gene_0,gene_1,gene_2,gene_3,gene_4,gene_5,gene_6,gene_7,gene_8,...,gene_20521,gene_20522,gene_20523,gene_20524,gene_20525,gene_20526,gene_20527,gene_20528,gene_20529,gene_20530
0,sample_0,0.0,2.017209,3.265527,5.478487,10.431999,0.0,7.175175,0.591871,0.0,...,4.926711,8.210257,9.723516,7.220030,9.119813,12.003135,9.650743,8.921326,5.286759,0.0
1,sample_1,0.0,0.592732,1.588421,7.586157,9.623011,0.0,6.816049,0.000000,0.0,...,4.593372,7.323865,9.740931,6.256586,8.381612,12.674552,10.517059,9.397854,2.094168,0.0
2,sample_2,0.0,3.511759,4.327199,6.881787,9.870730,0.0,6.972130,0.452595,0.0,...,5.125213,8.127123,10.908640,5.401607,9.911597,9.045255,9.788359,10.090470,1.683023,0.0
3,sample_3,0.0,3.663618,4.507649,6.659068,10.196184,0.0,7.843375,0.434882,0.0,...,6.076566,8.792959,10.141520,8.942805,9.601208,11.392682,9.694814,9.684365,3.292001,0.0
4,sample_4,0.0,2.655741,2.821547,6.539454,9.738265,0.0,6.566967,0.360982,0.0,...,5.996032,8.891425,10.373790,7.181162,9.846910,11.922439,9.217749,9.461191,5.110372,0.0


In [23]:
label_df = pd.read_csv('/kaggle/input/gene-expression-cancer-rna-seq/TCGA-PANCAN-HiSeq-801x20531/TCGA-PANCAN-HiSeq-801x20531/labels.csv')
label_df.head()

,Unnamed: 0,Class
0,sample_0,PRAD
1,sample_1,LUAD
2,sample_2,PRAD
3,sample_3,PRAD
4,sample_4,BRCA


### Shape of Data and Columns

In [24]:
print("The Data Set has 2 parts")
print("Data.csv")
print("    Shape: ", dataDF.shape)
print("    Columns: ", dataDF.columns)
print("Label.csv")
print("    Shape: ", labelDF.shape)
print("    Columns: ", labelDF.columns)

The Data Set has 2 parts
Data.csv
    Shape:  (801, 20532)
    Columns:  Index(['Unnamed: 0', 'gene_0', 'gene_1', 'gene_2', 'gene_3', 'gene_4',
       'gene_5', 'gene_6', 'gene_7', 'gene_8',
       ...
       'gene_20521', 'gene_20522', 'gene_20523', 'gene_20524', 'gene_20525',
       'gene_20526', 'gene_20527', 'gene_20528', 'gene_20529', 'gene_20530'],
      dtype='object', length=20532)
Label.csv
    Shape:  (801, 2)
    Columns:  Index(['Unnamed: 0', 'Class'], dtype='object')


## Data Preprocessing

In [25]:
# Drop the unwanted "Unnamed: 0" column
data_df = data_df.drop(columns=['Unnamed: 0'])
label_df = label_df.drop(columns=['Unnamed: 0'])

In [26]:
data_df.sample(1)

,gene_0,gene_1,gene_2,gene_3,gene_4,gene_5,gene_6,gene_7,gene_8,gene_9,...,gene_20521,gene_20522,gene_20523,gene_20524,gene_20525,gene_20526,gene_20527,gene_20528,gene_20529,gene_20530
590,0.0,3.191168,2.927896,7.776644,10.637431,0.0,7.102007,1.156138,0.0,0.0,...,5.563106,8.655248,9.463998,0.495183,9.673131,12.597822,9.793197,9.748562,4.119497,0.0


## CLIQUE (pyclustering)

In [27]:
!pip install pyclustering

In [28]:
from pyclustering.cluster.clique import clique
from pyclustering.utils import read_sample
from pyclustering.samples.definitions import SIMPLE_SAMPLES
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import random

# Normalize the data for CLIQUE (important for grid-based clustering)
scaler = MinMaxScaler()
data_normalized = scaler.fit_transform(data_df)

# Store gene indices that participate in clusters
genes_in_clusters_across_dims = []

# Try 50 random 2D projections
num_iterations = 50
n_genes = data_df.shape[1]
gene_indices = list(range(n_genes))

for _ in range(num_iterations):
    # Randomly select 2 gene indices
    gene_pair = random.sample(gene_indices, 2)
    data_2d = data_normalized[:, gene_pair]

    # Apply CLIQUE
    clique_instance = clique(data_2d.tolist(), 10, 0.1)
    clique_instance = clique(data_2d, 10, 2)
    clique_instance.process()
    clusters = clique_instance.get_clusters()

    # If there are valid clusters, store gene indices
    if any(len(cluster) > 1 for cluster in clusters):
        genes_in_clusters_across_dims.append(set(gene_pair))


## Identify Key Genes

In [29]:
# Take intersection across all clustering dimensions
if genes_in_clusters_across_dims:
    key_genes_indices = set.intersection(*genes_in_clusters_across_dims)
    key_genes = [data_df.columns[i] for i in key_genes_indices]
    print(f"Identified {len(key_genes)} key genes.")
    print("Key Genes:", key_genes)
else:
    print("No overlapping genes found across dimensions.")


Identified 0 key genes.
Key Genes: []


### Save OUTPUT

In [30]:
pd.Series(key_genes).to_csv("/kaggle/working/key_genes.csv", index=False)